In [1]:
from datetime import datetime

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [2]:
print(get_current_datetime())          
print(get_current_datetime("%H:%M"))    

2026-06-26 14:24:33
14:24


In [3]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format. Use this when you need to know the current date or time. Returns a string with the formatted datetime.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

In [4]:
import anthropic

In [5]:
client = anthropic.Anthropic()

In [6]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

In [7]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print(response.content) 

[ToolUseBlock(id='toolu_017vcKkJ99FEX7pizzZb2Sui', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]


In [8]:
messages.append({
    "role": "assistant",
    "content": response.content  
})

## What Happens After Claude Requests a Tool?
You extract the parameters Claude wants, run the actual function, then send the result back in a special **tool result block**.

## Tool Result Block : 3 Key Fields
- **tool_use_id** : must match the ID from Claude's ToolUse block so Claude knows which result belongs to which request
- **content** : the output of your function, as a string
- **is_error** : set to `True` if something went wrong

In [9]:
tool_input = response.content[0].input
result = get_current_datetime(**tool_input)
print(result)

14:24:36


In [10]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": result,
        "is_error": False
    }]
})

final_response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema] 
)

print(final_response.content)

[TextBlock(citations=None, text='The exact current time is **14:24:36** (in 24-hour format, HH:MM:SS).', type='text')]
